In [12]:
# Импорт библиотек
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras import layers, models

# Параметры для аугментации данных
train_datagen = ImageDataGenerator(
    rescale=1./255,               # Нормализация изображений
    rotation_range=20,            # Вращение изображений
    width_shift_range=0.2,        # Сдвиг по ширине
    height_shift_range=0.2,       # Сдвиг по высоте
    shear_range=0.2,              # Сдвиг
    zoom_range=0.2,               # Увеличение/уменьшение изображения
    horizontal_flip=True,         # Горизонтальное отражение
    fill_mode='nearest'           # Метод заполнения при преобразованиях
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Генератор для тренировки с масками
def create_segmentation_generators(image_dir, mask_dir=None, batch_size=16, target_size=(224, 224)):
    image_filenames = os.listdir(image_dir)
    image_filenames.sort()

    while True:  # Бесконечный цикл для генератора
        for i in range(0, len(image_filenames), batch_size):
            batch_images = []
            
            # Загружаем изображения для текущего батча
            for j in range(i, min(i + batch_size, len(image_filenames))):
                img_path = os.path.join(image_dir, image_filenames[j])
                
                img = load_img(img_path, target_size=target_size)
                img_array = img_to_array(img) / 255.0  # Нормализация
                
                batch_images.append(img_array)
            
            # Преобразуем в numpy массив и возвращаем
            if mask_dir:  # Если маски предоставлены, загружаем их
                mask_filenames = os.listdir(mask_dir)
                mask_filenames.sort()
                batch_masks = []

                for j in range(i, min(i + batch_size, len(mask_filenames))):
                    mask_path = os.path.join(mask_dir, mask_filenames[j])
                    mask = load_img(mask_path, target_size=target_size, color_mode='grayscale')
                    mask_array = img_to_array(mask) / 255.0  # Нормализация маски
                    batch_masks.append(mask_array)

                yield np.array(batch_images), np.array(batch_masks)
            else:  # Если нет масок, возвращаем только изображения
                yield np.array(batch_images)

# Генератор для тренировочных данных (с масками)
train_generator = create_segmentation_generators(
    'data/TRAINING_IMAGES',  # Путь к папке с тренировочными изображениями
    'data/TRAINING_MASKS',   # Путь к папке с тренировочными масками
    batch_size=16
)

# Генератор для валидационных данных (без масок)
val_generator = create_segmentation_generators(
    'data/VAL_IMAGES',  # Путь к папке с валидационными изображениями
    batch_size=16
)


In [13]:
def unet_model(input_size=(224, 224, 3)):
    inputs = layers.Input(input_size)
    
    # Энкодер
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # Боттлнек
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)

    # Декодер
    u6 = layers.UpSampling2D((2, 2))(c5)
    u6 = layers.concatenate([u6, c4], axis=-1)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c6)

    u7 = layers.UpSampling2D((2, 2))(c6)
    u7 = layers.concatenate([u7, c3], axis=-1)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c7)

    u8 = layers.UpSampling2D((2, 2))(c7)
    u8 = layers.concatenate([u8, c2], axis=-1)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c8)

    u9 = layers.UpSampling2D((2, 2))(c8)
    u9 = layers.concatenate([u9, c1], axis=-1)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c9)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)

    model = models.Model(inputs, outputs)
    return model

model = unet_model()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])


In [15]:
# Обучение модели
history = model.fit(
    train_generator,
    steps_per_epoch=len(os.listdir('data/TRAINING_IMAGES')) // 16,  # Количество шагов на эпоху
    epochs=20,
    validation_data=val_generator,
    validation_steps=len(os.listdir('data/VAL_IMAGES')) // 16   # Количество шагов на эпоху для валидации
)


Epoch 1/20
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step - accuracy: 0.8662 - loss: 0.3475 

IndexError: list index out of range

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    'data/test',  # Путь к папке с тестовыми данными
    target_size=(224, 224),
    batch_size=16,
    class_mode=None,
    seed=42
)

# Оценка модели на тестовых данных
model.evaluate(test_generator, steps=len(test_generator))


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

# Загрузим изображение
img_path = 'data/test_image.jpg'
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0  # Нормализация
img_array = np.expand_dims(img_array, axis=0)

# Получаем предсказание
pred = model.predict(img_array)

# Отображаем результат
import matplotlib.pyplot as plt

plt.imshow(pred[0, :, :, 0], cmap='gray')
plt.show()
